# Direct-ML (no stage1 hurdle) experiment — Reservoir Inflow (เวอร์ชันรันบนเครื่อง)

ทดลอง: เทรน ML ทำนาย Q_in ตรงๆ (delta จาก Q_in_t ปัจจุบัน) ทุกแถวรวมกัน **ไม่มี stage1 classifier
กรอง zero/nonzero** เทียบ 3 โมเดล (CatBoost/XGBoost/LightGBM), Optuna tune ต่อโมเดลต่อ horizon

**เวอร์ชันนี้ (รอบ 3, 2026-07-26)**: ใช้ `Training_Values_Nofct_7day_Extended_lagfeat2.csv` ซึ่งเพิ่มจากไฟล์ต้นฉบับ
`Training_Values_Nofct_7day_Extended.csv` สองรอบ:
1. `Qin_lag1`/`Qin_lag2` (Q_in ย้อนหลัง 1-2 วัน) + `Rain_roll3`/`Rain_roll5`/`Rain_roll7` (ฝนสะสมย้อนหลัง)
   → ผลลัพธ์: walk-forward CV NSE ดีขึ้น 5/7 horizon, H3 ดีขึ้นมากที่สุด (-0.3719 → -0.0649)
2. `WaterLevel_lag1/2`, `Storage_lag1/2`, `PctFull_lag1/2` (สถานะอ่างเก็บน้ำย้อนหลัง) — เผื่อช่วยจับ
   pattern การปล่อยน้ำ/คุมเขื่อนที่ไม่สัมพันธ์กับฝนตรงๆ (เช่นคลัสเตอร์ 4-6 ม.ค. ที่ฝน 0mm แต่ Qin สวิงแรง)

**Checkpoint/resume**: ผลลัพธ์เขียนลง `outputs_local_lagfeat2/*.csv` ทีละ (Model, Horizon) ถ้าปิด kernel/รันซ้ำ
กลางทาง แค่รัน cell ให้ครบใหม่ มันจะข้ามส่วนที่เสร็จแล้วเอง

โครงสร้างไฟล์นี้: Cell 1 ติดตั้ง lib | Cell 2 setup | Cell 3 โหลด+split | Cell 4 tuning (N_TRIALS=100) |
Cell 5 walk-forward CV | Cell 6 สรุปเทียบ production | Cell 7 export .pkl | Cell 8 วินิจฉัย H3

## Cell 1 — ติดตั้ง library (รันครั้งเดียว ข้ามได้ถ้าติดตั้งไว้แล้ว)

ถ้าใช้ venv/conda แยกอยู่แล้ว รันใน terminal แทนก็ได้: `pip install catboost xgboost lightgbm optuna pandas numpy scikit-learn`

In [1]:
import sys
!{sys.executable} -m pip install catboost xgboost lightgbm optuna pandas numpy scikit-learn --quiet

## Cell 2 — Setup: imports, model adapters, Optuna search space, metric functions

In [2]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTDIR = "outputs_local_lagfeat2"
os.makedirs(OUTDIR, exist_ok=True)
PERF_CSV = os.path.join(OUTDIR, "all_models_test_performance.csv")
PARAMS_CSV = os.path.join(OUTDIR, "all_models_tuned_params.csv")
CV_CSV = os.path.join(OUTDIR, "cv_stability_diagnostic.csv")
CSV_PATH = "Training_Values_Nofct_7day_Extended_lagfeat2.csv"  # ต้องอยู่โฟลเดอร์เดียวกับ notebook นี้

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb


class CatBoostAdapter:
    name = "CatBoost"
    @staticmethod
    def build(params, with_early_stopping=True):
        p = dict(params); p.setdefault("loss_function", "MAE"); p.setdefault("bootstrap_type", "Bernoulli")
        p.setdefault("random_seed", 42); p.setdefault("verbose", False)
        if with_early_stopping: p.setdefault("od_type", "Iter"); p.setdefault("od_wait", 200)
        return CatBoostRegressor(**p)
    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
            return model, model.get_best_iteration()
        model.fit(X_tr, y_tr, verbose=False); return model, model.tree_count_
    @staticmethod
    def predict(model, X): return model.predict(X)


class XGBoostAdapter:
    name = "XGBoost"
    @staticmethod
    def build(params, with_early_stopping=True):
        p = dict(params); p.setdefault("random_state", 42); p.setdefault("verbosity", 0)
        if with_early_stopping: p.setdefault("early_stopping_rounds", 50); p.setdefault("eval_metric", "rmse")
        return XGBRegressor(**p)
    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            bi = model.best_iteration if hasattr(model, "best_iteration") and model.best_iteration is not None else model.n_estimators
            return model, bi
        model.fit(X_tr, y_tr, verbose=False); return model, model.n_estimators
    @staticmethod
    def predict(model, X): return model.predict(X)


class LightGBMAdapter:
    name = "LightGBM"
    @staticmethod
    def build(params, with_early_stopping=True):
        p = dict(params); p.setdefault("random_state", 42); p.setdefault("verbose", -1)
        return LGBMRegressor(**p)
    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="rmse",
                       callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
            return model, model.best_iteration_
        model.fit(X_tr, y_tr); return model, model.n_estimators_
    @staticmethod
    def predict(model, X): return model.predict(X)


MODEL_ADAPTERS = {"CatBoost": CatBoostAdapter, "XGBoost": XGBoostAdapter, "LightGBM": LightGBMAdapter}


def make_optuna_params(trial, model_name):
    if model_name == "CatBoost":
        return dict(iterations=trial.suggest_int("iterations", 300, 1500, step=50),
                     learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
                     depth=trial.suggest_int("depth", 2, 6),
                     l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 500.0, log=True),
                     min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 5, 40),
                     subsample=trial.suggest_float("subsample", 0.4, 1.0),
                     rsm=trial.suggest_float("rsm", 0.5, 1.0),
                     random_strength=trial.suggest_float("random_strength", 0.0, 3.0))
    elif model_name == "XGBoost":
        return dict(n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=50),
                     learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
                     max_depth=trial.suggest_int("max_depth", 2, 6),
                     reg_lambda=trial.suggest_float("reg_lambda", 1.0, 500.0, log=True),
                     min_child_weight=trial.suggest_int("min_child_weight", 1, 40),
                     subsample=trial.suggest_float("subsample", 0.4, 1.0),
                     colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0))
    elif model_name == "LightGBM":
        return dict(n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=50),
                     learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
                     max_depth=trial.suggest_int("max_depth", 2, 6),
                     reg_lambda=trial.suggest_float("reg_lambda", 1.0, 500.0, log=True),
                     min_child_samples=trial.suggest_int("min_child_samples", 5, 40),
                     subsample=trial.suggest_float("subsample", 0.4, 1.0),
                     colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                     subsample_freq=1)
    raise ValueError(model_name)


def mae_np(y, yhat): y, yhat = np.asarray(y), np.asarray(yhat); return float(np.mean(np.abs(y - yhat)))
def rmse_np(y, yhat): y, yhat = np.asarray(y), np.asarray(yhat); return float(np.sqrt(np.mean((y - yhat) ** 2)))
def nse_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(1 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-9))
def kge_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    r = np.corrcoef(y, yhat)[0, 1]
    alpha = (yhat.std() + 1e-9) / (y.std() + 1e-9); beta = (yhat.mean() + 1e-9) / (y.mean() + 1e-9)
    return float(1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))

print("Setup OK")

Setup OK


## Cell 3 — Part 1: โหลดข้อมูล + chronological wet/dry/wet split (70/15/15)

In [3]:
df = pd.read_csv(CSV_PATH)
df["Date_dt"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values("Date_dt").reset_index(drop=True)
print("Date range:", df["Date_dt"].min(), "to", df["Date_dt"].max(), "| Rows:", len(df))

QIN_COL = "Q_in_t (m3/day)"
targets = sorted([c for c in df.columns if re.match(r"^y\d+=Q_in", str(c))],
                  key=lambda x: int(re.findall(r"^y(\d+)=", x)[0]))
print("Targets:", targets)
meta_cols = {"manual_copy_paste_error_corrected", "correction_note", "data_lineage_uncertain",
             "lineage_group", "training_value_replaced_no_duplicate_evidence", "raw_diverged_from_training_snapshot"}
exclude = {"Date", "Date_num", "Date_dt"} | set(targets) | meta_cols
feature_cols = [c for c in df.columns if c not in exclude]
print("Features:", feature_cols)
n_nan = df["DeltaS_t (m3/day)"].isna().sum()
if n_nan: print(f"Forward-filling {n_nan} NaN in DeltaS_t")
df["DeltaS_t (m3/day)"] = df["DeltaS_t (m3/day)"].ffill()

X = df[feature_cols].copy(); Y = df[targets].copy()
mask = (~Y.isna().any(axis=1)) & (~df["Date_dt"].isna())
X, Y = X.loc[mask].reset_index(drop=True), Y.loc[mask].reset_index(drop=True)
dates = df.loc[mask, "Date_dt"].reset_index(drop=True)
N = len(dates)
print("Rows used:", N)

ZERO_THRESHOLD = 1e-6
dry_start, dry_end = pd.Timestamp("2026-02-03"), pd.Timestamp("2026-05-17")
wet_before_idx = np.where(dates < dry_start)[0]
dry_idx = np.where((dates >= dry_start) & (dates <= dry_end))[0]
wet_after_idx = np.where(dates > dry_end)[0]
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
n_dry = len(dry_idx); n_dry_train = round(n_dry * TRAIN_FRAC); n_dry_val = round(n_dry * VAL_FRAC)
dry_train_idx = dry_idx[:n_dry_train]; dry_val_idx = dry_idx[n_dry_train:n_dry_train + n_dry_val]
dry_test_idx = dry_idx[n_dry_train + n_dry_val:]

def chrono_split(idx_array, frac_train=TRAIN_FRAC, frac_val=VAL_FRAC):
    n = len(idx_array); c1 = int(round(n * frac_train)); c2 = int(round(n * (frac_train + frac_val)))
    return idx_array[:c1], idx_array[c1:c2], idx_array[c2:]

wb_train, wb_val, wb_test = chrono_split(wet_before_idx)
wa_train, wa_val, wa_test = chrono_split(wet_after_idx)
train_idx = np.sort(np.concatenate([wb_train, dry_train_idx, wa_train]))
val_idx = np.sort(np.concatenate([wb_val, dry_val_idx, wa_val]))
test_idx = np.sort(np.concatenate([wb_test, dry_test_idx, wa_test]))

X_train, Y_train_abs = X.iloc[train_idx].reset_index(drop=True), Y.iloc[train_idx].reset_index(drop=True)
X_val, Y_val_abs = X.iloc[val_idx].reset_index(drop=True), Y.iloc[val_idx].reset_index(drop=True)
X_test, Y_test_abs = X.iloc[test_idx].reset_index(drop=True), Y.iloc[test_idx].reset_index(drop=True)
dates_test = dates.iloc[test_idx].reset_index(drop=True)
base_test = X_test[QIN_COL].to_numpy()
print(f"Split sizes: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

Date range: 2025-06-27 00:00:00 to 2026-07-24 00:00:00 | Rows: 393
Targets: ['y1=Q_in_t+1 (m3/day)', 'y2=Q_in_t+2 (m3/day)', 'y3=Q_in_t+3 (m3/day)', 'y4=Q_in_t+4 (m3/day)', 'y5=Q_in_t+5 (m3/day)', 'y6=Q_in_t+6 (m3/day)', 'y7=Q_in_t+7 (m3/day)']
Features: ['Q_in_t (m3/day)', 'Water_Level_t (m)', 'Storage_S_t (m3)', 'DeltaS_t (m3/day)', '%Full_t', 'Rain_obs_t (mm)', 'API_t (mm)', 'Qin_lag1 (m3/day)', 'Qin_lag2 (m3/day)', 'Rain_roll3 (mm)', 'Rain_roll5 (mm)', 'Rain_roll7 (mm)', 'WaterLevel_lag1', 'WaterLevel_lag2', 'Storage_lag1', 'Storage_lag2', 'PctFull_lag1', 'PctFull_lag2']
Forward-filling 1 NaN in DeltaS_t
Rows used: 386
Split sizes: train=271, val=58, test=57


## Cell 4 — Part 2: Optuna tuning ต่อโมเดลต่อ horizon (checkpointed)

N_TRIALS=100 — เครื่อง i7-12700H แรงพอลองสูงกว่านี้ได้อีกถ้าต้องการ

In [ ]:
N_TRIALS = 120
INNER_SPLITS = 4

def make_objective(model_name, X_tr, y_tr_delta, qin_tr):
    tscv_inner = TimeSeriesSplit(n_splits=INNER_SPLITS)
    adapter = MODEL_ADAPTERS[model_name]
    def objective(trial):
        params = make_optuna_params(trial, model_name)
        fold_rmses = []
        for tr_idx, te_idx in tscv_inner.split(X_tr):
            Xtr_f, Xte_f = X_tr.iloc[tr_idx], X_tr.iloc[te_idx]
            ytr_f, yte_f = y_tr_delta[tr_idx], y_tr_delta[te_idx]
            qin_te_f = qin_tr[te_idx]
            m = adapter.build(params, with_early_stopping=False)
            m, _ = adapter.fit(m, Xtr_f, ytr_f)
            y_pred_abs_f = np.clip(qin_te_f + adapter.predict(m, Xte_f), 0, None)
            y_true_abs_f = yte_f + qin_te_f
            fold_rmses.append(rmse_np(y_true_abs_f, y_pred_abs_f))
        return float(np.mean(fold_rmses))
    return objective


done_pairs = set()
if os.path.exists(PERF_CSV):
    prev = pd.read_csv(PERF_CSV)
    done_pairs = set(zip(prev["Model"], prev["H"]))
    print(f"Resuming: {len(done_pairs)} combos already done")

t0 = time.time()
all_combos = [(mn, h, tcol) for mn in MODEL_ADAPTERS for h, tcol in enumerate(targets, start=1)]

for model_name, h, tcol in all_combos:
    if (model_name, h) in done_pairs:
        continue
    adapter = MODEL_ADAPTERS[model_name]
    qin_train_arr = X_train[QIN_COL].to_numpy()
    y_train_delta = Y_train_abs[tcol].to_numpy() - qin_train_arr

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(model_name, X_train, y_train_delta, qin_train_arr), n_trials=N_TRIALS)
    best_params = study.best_params

    yval_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
    m = adapter.build(best_params, with_early_stopping=True)
    m, best_iter = adapter.fit(m, X_train, y_train_delta, X_val, yval_delta)

    delta_pred_test = adapter.predict(m, X_test)
    q_pred_test = np.clip(base_test + delta_pred_test, 0, None)
    y_true_test = Y_test_abs[tcol].to_numpy()
    delta_pred_val = adapter.predict(m, X_val)
    q_pred_val = np.clip(X_val[QIN_COL].to_numpy() + delta_pred_val, 0, None)

    perf_row = {
        "Model": model_name, "H": h, "Target": tcol, "best_iter": best_iter,
        "inner_cv_rmse": study.best_value,
        "Baseline_MAE": mae_np(y_true_test, base_test), "Test_MAE": mae_np(y_true_test, q_pred_test),
        "Baseline_RMSE": rmse_np(y_true_test, base_test), "Test_RMSE": rmse_np(y_true_test, q_pred_test),
        "Baseline_NSE": nse_np(y_true_test, base_test), "Test_NSE": nse_np(y_true_test, q_pred_test),
        "Test_KGE": kge_np(y_true_test, q_pred_test),
        "Val_NSE": nse_np(Y_val_abs[tcol].to_numpy(), q_pred_val),
    }
    params_row = {"Model": model_name, "H": h, "Target": tcol, "inner_cv_rmse": study.best_value, **best_params}
    pd.DataFrame([perf_row]).to_csv(PERF_CSV, mode="a", header=not os.path.exists(PERF_CSV), index=False)
    pd.DataFrame([params_row]).to_csv(PARAMS_CSV, mode="a", header=not os.path.exists(PARAMS_CSV), index=False)
    print(f"[{time.time()-t0:.0f}s] {model_name} H{h}: Test_NSE={perf_row['Test_NSE']:.4f} "
          f"(baseline={perf_row['Baseline_NSE']:.4f}), Val_NSE={perf_row['Val_NSE']:.4f}")

print(f"\nPart 2 done, {time.time()-t0:.0f}s total")

## Cell 5 — Part 3: walk-forward CV diagnostic (checkpointed)

ใช้ tuned params จาก Cell 4 ประเมินความเสถียรแบบ TimeSeriesSplit 5-fold บน train block เดียวกับที่
metric `walkforward_cv_nse` บนเว็บใช้อยู่ตอนนี้ -- ต้องรัน Cell 4 ให้เสร็จ (ครบ 21/21 combos) ก่อน

In [ ]:
tuned_params_df = pd.read_csv(PARAMS_CSV)
int_cols = ["iterations", "n_estimators", "depth", "max_depth", "min_data_in_leaf",
            "min_child_weight", "min_child_samples", "subsample_freq"]

done_pairs_cv = set()
if os.path.exists(CV_CSV):
    prev = pd.read_csv(CV_CSV)
    done_pairs_cv = set(zip(prev["Model"], prev["H"]))
    print(f"Resuming: {len(done_pairs_cv)} combos already done")

t1 = time.time()
tscv = TimeSeriesSplit(n_splits=5)
all_combos = [(mn, h, tcol) for mn in MODEL_ADAPTERS for h, tcol in enumerate(targets, start=1)]

for model_name, h, tcol in all_combos:
    if (model_name, h) in done_pairs_cv:
        continue
    adapter = MODEL_ADAPTERS[model_name]
    tuned_row = tuned_params_df[(tuned_params_df["Model"] == model_name) & (tuned_params_df["H"] == h)].iloc[0]
    param_cols = [c for c in tuned_params_df.columns if c not in ("Model", "H", "Target", "inner_cv_rmse")]
    params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
    for k in int_cols:
        if k in params:
            params[k] = int(params[k])

    fold_rows = []
    for fold, (tr, te) in enumerate(tscv.split(X_train), start=1):
        Xtr_cv, Xte_cv = X_train.iloc[tr].reset_index(drop=True), X_train.iloc[te].reset_index(drop=True)
        base_te_cv = Xte_cv[QIN_COL].to_numpy()
        ytr_abs_cv = Y_train_abs[tcol].iloc[tr].reset_index(drop=True).to_numpy()
        yte_abs_cv = Y_train_abs[tcol].iloc[te].reset_index(drop=True).to_numpy()
        ytr_delta_cv = ytr_abs_cv - Xtr_cv[QIN_COL].to_numpy()
        m_cv = adapter.build(params, with_early_stopping=False)
        m_cv, _ = adapter.fit(m_cv, Xtr_cv, ytr_delta_cv)
        yhat_cv = np.clip(base_te_cv + adapter.predict(m_cv, Xte_cv), 0, None)
        fold_rows.append({"Model": model_name, "fold": fold, "H": h,
                           "Baseline_NSE": nse_np(yte_abs_cv, base_te_cv),
                           "Model_NSE": nse_np(yte_abs_cv, yhat_cv)})

    pd.DataFrame(fold_rows).to_csv(CV_CSV, mode="a", header=not os.path.exists(CV_CSV), index=False)
    mean_nse = np.mean([r["Model_NSE"] for r in fold_rows])
    print(f"[{time.time()-t1:.0f}s] {model_name} H{h}: walkforward_cv_NSE mean={mean_nse:.4f}")

print(f"\nPart 3 done, {time.time()-t1:.0f}s total")

## Cell 6 — สรุปเทียบกับ production ปัจจุบัน

ตัวเลข `CURRENT_PROD_WF_CV_NSE` ก็อปมาจาก `active/model_metadata.json` ตอนทำ experiment
(2026-07-26) ถ้า production เปลี่ยนไปแล้วให้ไปเอาตัวเลขล่าสุดมาแทน

In [ ]:
CURRENT_PROD_WF_CV_NSE = {1: 0.4785, 2: -0.0312, 3: -1.6972, 4: -1.8345, 5: -3.528, 6: -15.3064, 7: -13.2143}

cv_df = pd.read_csv(CV_CSV)
summary = cv_df.groupby(["Model", "H"])["Model_NSE"].mean().reset_index()
pivot = summary.pivot(index="H", columns="Model", values="Model_NSE")
print("=== walk-forward CV NSE (mean of 5 folds) by model x horizon ===")
print(pivot.round(4).to_string())

print("\n=== FINAL: current production (hurdle) vs best new direct-ML model per horizon ===")
print(f"{'H':>3} {'current_hurdle':>15} {'best_new_model':>16} {'best_new_nse':>14} {'winner':>10}")
for h in range(1, len(targets) + 1):
    cur = CURRENT_PROD_WF_CV_NSE[h]
    row = pivot.loc[h]
    best_model = row.idxmax()
    best_nse = row.max()
    winner = "NEW" if best_nse > cur else "current"
    print(f"{h:>3} {cur:>15.4f} {best_model:>16} {best_nse:>14.4f} {winner:>10}")

## Cell 7 — Export โมเดลรอบนี้เป็น .pkl (เช็คพอยต์ผลปัจจุบัน ก่อนไปปรับ feature/อื่นๆ ต่อ)

รันต่อจาก Cell 2-5 ในเคอร์เนลเดิมได้เลย (ใช้ `X_train`/`X_val`/`MODEL_ADAPTERS`/`targets` ที่มีอยู่แล้ว
ในหน่วยความจำ) **ถ้า restart kernel มาใหม่ ก็รันเซลนี้เดี่ยวๆ ได้เลย** จะโหลด+split ข้อมูลใหม่ให้เอง
(ไม่ต้องรัน Cell 4/5 ใหม่ เพราะอ่าน tuned params จาก `outputs_local_lagfeat2/*.csv` ที่บันทึกไว้แล้ว)

โมเดลที่ export = fit ด้วย best params บน X_train + early stopping ด้วย X_val -- fit แบบเดียวกับที่
Cell 4 ใช้ตอนสร้างเลข Test_NSE/Val_NSE ที่รายงานไปแล้วทุกประการ เลือกโมเดลที่ดีที่สุดต่อ horizon จาก
walk-forward CV NSE (ไม่ใช่ Test_NSE เพราะ test set มีแค่ 57 แถว วัดผลไม่นิ่ง)

In [ ]:
import os, re, pickle, json
from datetime import datetime
import numpy as np
import pandas as pd

OUTDIR = "outputs_local_lagfeat2"
PARAMS_CSV = os.path.join(OUTDIR, "all_models_tuned_params.csv")
CV_CSV = os.path.join(OUTDIR, "cv_stability_diagnostic.csv")
PERF_CSV = os.path.join(OUTDIR, "all_models_test_performance.csv")

EXPORT_DIR = "exported_" + datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(EXPORT_DIR, exist_ok=True)

if "X_train" not in dir():
    print("X_train ไม่พบในหน่วยความจำ -- โหลด+split ข้อมูลใหม่ (เหมือน Cell 3)")
    CSV_PATH = "Training_Values_Nofct_7day_Extended_lagfeat2.csv"
    df = pd.read_csv(CSV_PATH)
    df["Date_dt"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date_dt").reset_index(drop=True)
    QIN_COL = "Q_in_t (m3/day)"
    targets = sorted([c for c in df.columns if re.match(r"^y\d+=Q_in", str(c))],
                      key=lambda x: int(re.findall(r"^y(\d+)=", x)[0]))
    meta_cols = {"manual_copy_paste_error_corrected", "correction_note", "data_lineage_uncertain",
                 "lineage_group", "training_value_replaced_no_duplicate_evidence", "raw_diverged_from_training_snapshot"}
    exclude = {"Date", "Date_num", "Date_dt"} | set(targets) | meta_cols
    feature_cols = [c for c in df.columns if c not in exclude]
    df["DeltaS_t (m3/day)"] = df["DeltaS_t (m3/day)"].ffill()
    X = df[feature_cols].copy(); Y = df[targets].copy()
    mask = (~Y.isna().any(axis=1)) & (~df["Date_dt"].isna())
    X, Y = X.loc[mask].reset_index(drop=True), Y.loc[mask].reset_index(drop=True)
    dates = df.loc[mask, "Date_dt"].reset_index(drop=True)
    dry_start, dry_end = pd.Timestamp("2026-02-03"), pd.Timestamp("2026-05-17")
    wet_before_idx = np.where(dates < dry_start)[0]
    dry_idx = np.where((dates >= dry_start) & (dates <= dry_end))[0]
    wet_after_idx = np.where(dates > dry_end)[0]
    TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
    n_dry = len(dry_idx); n_dry_train = round(n_dry * TRAIN_FRAC); n_dry_val = round(n_dry * VAL_FRAC)
    dry_train_idx = dry_idx[:n_dry_train]; dry_val_idx = dry_idx[n_dry_train:n_dry_train + n_dry_val]
    dry_test_idx = dry_idx[n_dry_train + n_dry_val:]
    def chrono_split(idx_array, frac_train=TRAIN_FRAC, frac_val=VAL_FRAC):
        n = len(idx_array); c1 = int(round(n * frac_train)); c2 = int(round(n * (frac_train + frac_val)))
        return idx_array[:c1], idx_array[c1:c2], idx_array[c2:]
    wb_train, wb_val, wb_test = chrono_split(wet_before_idx)
    wa_train, wa_val, wa_test = chrono_split(wet_after_idx)
    train_idx = np.sort(np.concatenate([wb_train, dry_train_idx, wa_train]))
    val_idx = np.sort(np.concatenate([wb_val, dry_val_idx, wa_val]))
    test_idx = np.sort(np.concatenate([wb_test, dry_test_idx, wa_test]))
    X_train, Y_train_abs = X.iloc[train_idx].reset_index(drop=True), Y.iloc[train_idx].reset_index(drop=True)
    X_val, Y_val_abs = X.iloc[val_idx].reset_index(drop=True), Y.iloc[val_idx].reset_index(drop=True)
    X_test, Y_test_abs = X.iloc[test_idx].reset_index(drop=True), Y.iloc[test_idx].reset_index(drop=True)
    dates_test = dates.iloc[test_idx].reset_index(drop=True)

    from catboost import CatBoostRegressor
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
    import lightgbm as lgb

    class CatBoostAdapter:
        name = "CatBoost"
        @staticmethod
        def build(params, with_early_stopping=True):
            p = dict(params); p.setdefault("loss_function", "MAE"); p.setdefault("bootstrap_type", "Bernoulli")
            p.setdefault("random_seed", 42); p.setdefault("verbose", False)
            if with_early_stopping: p.setdefault("od_type", "Iter"); p.setdefault("od_wait", 200)
            return CatBoostRegressor(**p)
        @staticmethod
        def fit(model, X_tr, y_tr, X_val=None, y_val=None):
            if X_val is not None:
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
                return model, model.get_best_iteration()
            model.fit(X_tr, y_tr, verbose=False); return model, model.tree_count_
        @staticmethod
        def predict(model, X): return model.predict(X)

    class XGBoostAdapter:
        name = "XGBoost"
        @staticmethod
        def build(params, with_early_stopping=True):
            p = dict(params); p.setdefault("random_state", 42); p.setdefault("verbosity", 0)
            if with_early_stopping: p.setdefault("early_stopping_rounds", 50); p.setdefault("eval_metric", "rmse")
            return XGBRegressor(**p)
        @staticmethod
        def fit(model, X_tr, y_tr, X_val=None, y_val=None):
            if X_val is not None:
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
                bi = model.best_iteration if hasattr(model, "best_iteration") and model.best_iteration is not None else model.n_estimators
                return model, bi
            model.fit(X_tr, y_tr, verbose=False); return model, model.n_estimators
        @staticmethod
        def predict(model, X): return model.predict(X)

    class LightGBMAdapter:
        name = "LightGBM"
        @staticmethod
        def build(params, with_early_stopping=True):
            p = dict(params); p.setdefault("random_state", 42); p.setdefault("verbose", -1)
            return LGBMRegressor(**p)
        @staticmethod
        def fit(model, X_tr, y_tr, X_val=None, y_val=None):
            if X_val is not None:
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="rmse",
                           callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
                return model, model.best_iteration_
            model.fit(X_tr, y_tr); return model, model.n_estimators_
        @staticmethod
        def predict(model, X): return model.predict(X)

    MODEL_ADAPTERS = {"CatBoost": CatBoostAdapter, "XGBoost": XGBoostAdapter, "LightGBM": LightGBMAdapter}
else:
    print("ใช้ X_train/X_val/MODEL_ADAPTERS จากเคอร์เนลปัจจุบัน (ไม่ได้โหลดใหม่)")

tuned_params_df = pd.read_csv(PARAMS_CSV)
cv_df = pd.read_csv(CV_CSV)
perf_df = pd.read_csv(PERF_CSV)
cv_summary = cv_df.groupby(["Model", "H"])["Model_NSE"].mean().reset_index()

int_cols = ["iterations", "n_estimators", "depth", "max_depth", "min_data_in_leaf",
            "min_child_weight", "min_child_samples", "subsample_freq"]

fitted_models = {}
choice_meta = {}
for h in range(1, len(targets) + 1):
    tcol = targets[h - 1]
    row = cv_summary[cv_summary["H"] == h].sort_values("Model_NSE", ascending=False).iloc[0]
    model_name, wf_nse = row["Model"], row["Model_NSE"]
    adapter = MODEL_ADAPTERS[model_name]

    tuned_row = tuned_params_df[(tuned_params_df["Model"] == model_name) & (tuned_params_df["H"] == h)].iloc[0]
    param_cols = [c for c in tuned_params_df.columns if c not in ("Model", "H", "Target", "inner_cv_rmse")]
    params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
    for k in int_cols:
        if k in params:
            params[k] = int(params[k])

    y_train_delta = Y_train_abs[tcol].to_numpy() - X_train[QIN_COL].to_numpy()
    y_val_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
    m = adapter.build(params, with_early_stopping=True)
    m, best_iter = adapter.fit(m, X_train, y_train_delta, X_val, y_val_delta)
    fitted_models[h] = m

    perf_row = perf_df[(perf_df["Model"] == model_name) & (perf_df["H"] == h)].iloc[0]
    choice_meta[str(h)] = {
        "model_name": model_name,
        "target_column": tcol,
        "walkforward_cv_nse": float(wf_nse),
        "test_nse": float(perf_row["Test_NSE"]),
        "val_nse": float(perf_row["Val_NSE"]),
        "best_iter": int(best_iter),
        "tuned_params": {k: (int(v) if k in int_cols else float(v)) for k, v in params.items()},
    }
    print(f"H{h}: exported {model_name} (walkforward_cv_nse={wf_nse:.4f}, test_nse={perf_row['Test_NSE']:.4f})")

pkl_path = os.path.join(EXPORT_DIR, "deployment_regressors_no_stage1.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(fitted_models, f)

meta = {
    "exported_at": datetime.now().isoformat(),
    "architecture": "direct delta-regression per horizon, NO stage1 hurdle classifier "
                     "(different from current production which uses stage1 classifier + stage2 regressor)",
    "training_data_source": "Training_Values_Nofct_7day_Extended_lagfeat2.csv",
    "feature_cols": list(X_train.columns),
    "qin_col": QIN_COL,
    "targets": targets,
    "split_method": "chronological 70/15/15 within 3 season segments (wet_before/dry/wet_after), "
                     "dry window 2026-02-03 to 2026-05-17",
    "prediction_note": "predict delta from current Q_in_t, then q_pred = clip(Q_in_t + delta_pred, 0, None)",
    "deployment_model_per_horizon": choice_meta,
    "known_limitations": "H3-H5 walk-forward CV NSE still below current production's H1 only; "
                          "test set is small (57 rows) so Test_NSE per-horizon is noisy; see chat "
                          "discussion 2026-07-26 for diagnostics on train/test magnitude mismatch",
}
meta_path = os.path.join(EXPORT_DIR, "model_metadata_no_stage1.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\nExported to: {EXPORT_DIR}/")
print(f"  {pkl_path}")
print(f"  {meta_path}")
print("\nยังไม่ได้ wire เข้า production -- เป็นแค่ checkpoint ของผลรอบนี้ ปรับ feature/อื่นๆ ต่อได้โดยไม่กระทบไฟล์นี้")

## Cell 8 — วินิจฉัย error รายวันของ H3

Fit โมเดล H3 (CatBoost, best params เดิม) แล้วดู actual vs predicted ทีละวันใน test set (57 แถว)
เรียงจาก error มากไปน้อย พร้อมบริบท (ฝน, API, ระดับน้ำ) เพื่อดูว่าพลาดเพราะวันไหน/แบบไหน --
สร้างคอลัมน์ sq_err ด้วยเพราะ NSE คำนวณจาก sum of squared error พลาดวันเดียวแรงๆ ก็ฉุด NSE ทั้งก้อนได้

ใช้ตัวแปรจาก Cell 7 ได้เลยถ้ารันต่อในเคอร์เนลเดิม (ไม่มีก็ auto-reload เหมือนเดิม)

In [ ]:
import os, re
import numpy as np
import pandas as pd

OUTDIR = "outputs_local_lagfeat2"
PARAMS_CSV = os.path.join(OUTDIR, "all_models_tuned_params.csv")

if "X_train" not in dir():
    print("X_train ไม่พบในหน่วยความจำ -- โหลด+split ข้อมูลใหม่")
    CSV_PATH = "Training_Values_Nofct_7day_Extended_lagfeat2.csv"
    df = pd.read_csv(CSV_PATH)
    df["Date_dt"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date_dt").reset_index(drop=True)
    QIN_COL = "Q_in_t (m3/day)"
    targets = sorted([c for c in df.columns if re.match(r"^y\d+=Q_in", str(c))],
                      key=lambda x: int(re.findall(r"^y(\d+)=", x)[0]))
    meta_cols = {"manual_copy_paste_error_corrected", "correction_note", "data_lineage_uncertain",
                 "lineage_group", "training_value_replaced_no_duplicate_evidence", "raw_diverged_from_training_snapshot"}
    exclude = {"Date", "Date_num", "Date_dt"} | set(targets) | meta_cols
    feature_cols = [c for c in df.columns if c not in exclude]
    df["DeltaS_t (m3/day)"] = df["DeltaS_t (m3/day)"].ffill()
    X = df[feature_cols].copy(); Y = df[targets].copy()
    mask = (~Y.isna().any(axis=1)) & (~df["Date_dt"].isna())
    X, Y = X.loc[mask].reset_index(drop=True), Y.loc[mask].reset_index(drop=True)
    dates_all = df.loc[mask, "Date_dt"].reset_index(drop=True)
    dry_start, dry_end = pd.Timestamp("2026-02-03"), pd.Timestamp("2026-05-17")
    wet_before_idx = np.where(dates_all < dry_start)[0]
    dry_idx = np.where((dates_all >= dry_start) & (dates_all <= dry_end))[0]
    wet_after_idx = np.where(dates_all > dry_end)[0]
    TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
    n_dry = len(dry_idx); n_dry_train = round(n_dry * TRAIN_FRAC); n_dry_val = round(n_dry * VAL_FRAC)
    dry_train_idx = dry_idx[:n_dry_train]; dry_val_idx = dry_idx[n_dry_train:n_dry_train + n_dry_val]
    dry_test_idx = dry_idx[n_dry_train + n_dry_val:]
    def chrono_split(idx_array, frac_train=TRAIN_FRAC, frac_val=VAL_FRAC):
        n = len(idx_array); c1 = int(round(n * frac_train)); c2 = int(round(n * (frac_train + frac_val)))
        return idx_array[:c1], idx_array[c1:c2], idx_array[c2:]
    wb_train, wb_val, wb_test = chrono_split(wet_before_idx)
    wa_train, wa_val, wa_test = chrono_split(wet_after_idx)
    train_idx = np.sort(np.concatenate([wb_train, dry_train_idx, wa_train]))
    val_idx = np.sort(np.concatenate([wb_val, dry_val_idx, wa_val]))
    test_idx = np.sort(np.concatenate([wb_test, dry_test_idx, wa_test]))
    X_train, Y_train_abs = X.iloc[train_idx].reset_index(drop=True), Y.iloc[train_idx].reset_index(drop=True)
    X_val, Y_val_abs = X.iloc[val_idx].reset_index(drop=True), Y.iloc[val_idx].reset_index(drop=True)
    X_test, Y_test_abs = X.iloc[test_idx].reset_index(drop=True), Y.iloc[test_idx].reset_index(drop=True)
    dates_test = dates_all.iloc[test_idx].reset_index(drop=True)

    from catboost import CatBoostRegressor
    class CatBoostAdapter:
        name = "CatBoost"
        @staticmethod
        def build(params, with_early_stopping=True):
            p = dict(params); p.setdefault("loss_function", "MAE"); p.setdefault("bootstrap_type", "Bernoulli")
            p.setdefault("random_seed", 42); p.setdefault("verbose", False)
            if with_early_stopping: p.setdefault("od_type", "Iter"); p.setdefault("od_wait", 200)
            return CatBoostRegressor(**p)
        @staticmethod
        def fit(model, X_tr, y_tr, X_val=None, y_val=None):
            if X_val is not None:
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
                return model, model.get_best_iteration()
            model.fit(X_tr, y_tr, verbose=False); return model, model.tree_count_
        @staticmethod
        def predict(model, X): return model.predict(X)
    MODEL_ADAPTERS = {"CatBoost": CatBoostAdapter}
else:
    print("ใช้ X_train/X_val/X_test จากเคอร์เนลปัจจุบัน")
    if "dates_test" not in dir():
        dates_test = dates.iloc[test_idx].reset_index(drop=True)

H = 3
tcol = targets[H - 1]
tuned_params_df = pd.read_csv(PARAMS_CSV)
int_cols = ["iterations", "n_estimators", "depth", "max_depth", "min_data_in_leaf",
            "min_child_weight", "min_child_samples", "subsample_freq"]
tuned_row = tuned_params_df[(tuned_params_df["Model"] == "CatBoost") & (tuned_params_df["H"] == H)].iloc[0]
param_cols = [c for c in tuned_params_df.columns if c not in ("Model", "H", "Target", "inner_cv_rmse")]
params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
for k in int_cols:
    if k in params: params[k] = int(params[k])

adapter = MODEL_ADAPTERS["CatBoost"]
QIN_COL = "Q_in_t (m3/day)"
y_train_delta = Y_train_abs[tcol].to_numpy() - X_train[QIN_COL].to_numpy()
y_val_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
m = adapter.build(params, with_early_stopping=True)
m, _ = adapter.fit(m, X_train, y_train_delta, X_val, y_val_delta)

base_test = X_test[QIN_COL].to_numpy()
delta_pred = adapter.predict(m, X_test)
y_pred = np.clip(base_test + delta_pred, 0, None)
y_true = Y_test_abs[tcol].to_numpy()

diag = pd.DataFrame({
    "Date": dates_test.dt.date,
    "Qin_t (today)": base_test.round(0),
    "y3_actual": y_true.round(0),
    "y3_pred": y_pred.round(0),
    "baseline_pred(=Qin_t)": base_test.round(0),
    "model_error": (y_pred - y_true).round(0),
    "baseline_error": (base_test - y_true).round(0),
    "model_abs_err": np.abs(y_pred - y_true).round(0),
    "baseline_abs_err": np.abs(base_test - y_true).round(0),
    "model_sq_err": ((y_pred - y_true) ** 2),
    "Rain_obs_t": X_test["Rain_obs_t (mm)"].to_numpy(),
    "API_t": X_test["API_t (mm)"].to_numpy().round(1),
})
diag["model_beats_baseline"] = diag["model_abs_err"] < diag["baseline_abs_err"]
diag_sorted = diag.sort_values("model_sq_err", ascending=False)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print(f"=== H3 test set: {len(diag)} วัน, model ชนะ baseline {diag['model_beats_baseline'].sum()}/{len(diag)} วัน ===\n")
print("--- 10 วันที่ error (squared) แรงที่สุด (ตัวการหลักที่ฉุด NSE) ---")
print(diag_sorted.head(10).to_string(index=False))

total_sq = diag["model_sq_err"].sum()
top3_sq = diag_sorted.head(3)["model_sq_err"].sum()
print(f"\n3 วันแรกคิดเป็น {100*top3_sq/total_sq:.1f}% ของ sum-of-squared-error ทั้งหมด (จาก {len(diag)} วัน)")

print("\n--- สรุปเทียบ model vs baseline ---")
print(f"model sum_sq_err = {total_sq:,.0f}")
print(f"baseline sum_sq_err = {(diag['baseline_error']**2).sum():,.0f}")
print(f"model MAE = {diag['model_abs_err'].mean():,.0f} | baseline MAE = {diag['baseline_abs_err'].mean():,.0f}")